# `derived_8.4-ece-model-salvage-1.0`

This report evaluates seven retrained routing/model families after removing every SMAP-derived input. All fitting is restricted to the seven Washington training stations, while ECE targets are held out for spatial evaluation only.

## Reproducible setup

The next cell loads the tracked runner and generated artifacts so the notebook remains a reporting layer rather than a second implementation of the experiment.

In [1]:
from pathlib import Path
import importlib.util
import sys
import pandas as pd

EXP_DIR = Path("experiment/derived_8.4-ece-model-salvage-1.0").resolve()
if not (EXP_DIR / "run_model_salvage.py").exists():
    EXP_DIR = Path.cwd().resolve()
spec = importlib.util.spec_from_file_location("model_salvage", EXP_DIR / "run_model_salvage.py")
runner = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = runner
spec.loader.exec_module(runner)
config = runner.load_configuration()
predictions = pd.read_csv(EXP_DIR / "predictions.csv", low_memory=False)
seed_metrics = pd.read_csv(EXP_DIR / "seed_metrics.csv", low_memory=False)
summary = pd.read_csv(EXP_DIR / "summary.csv", low_memory=False)
audit = pd.read_csv(EXP_DIR / "routing_audit.csv", low_memory=False)
print(f"Experiment: {config['experiment']['name']}")
print(f"Models: {len(config['models'])}; seeds: {config['seeds']}")
print(f"Prediction rows: {len(predictions):,}")

Experiment: derived_8.4-ece-model-salvage-1.0
Models: 7; seeds: [42, 7, 13, 101, 123]
Prediction rows: 48,475


## Input and feature audit

This audit records the exact training/evaluation populations and confirms the effective feature counts after SMAP removal.

In [2]:
import json

print("REPORT_BEGIN::INPUT_AUDIT")
with (EXP_DIR / "input_audit.json").open(encoding="utf-8") as handle:
    print(json.dumps(json.load(handle), indent=2, sort_keys=True))
print("REPORT_END::INPUT_AUDIT")

print("REPORT_BEGIN::FEATURE_AUDIT")
with (EXP_DIR / "feature_manifest.json").open(encoding="utf-8") as handle:
    feature_manifest = json.load(handle)
feature_table = pd.DataFrame([
    {"component": name, "parent_count": len(value["parent"]), "dropped_smap": len(value["dropped"]), "effective_count": len(value["effective"]), "effective_features": ";".join(value["effective"])}
    for name, value in feature_manifest.items()
    if isinstance(value, dict) and "parent" in value
])
print(feature_table.to_markdown(index=False))
print("REPORT_END::FEATURE_AUDIT")

REPORT_BEGIN::INPUT_AUDIT
{
  "ece_date_max": "2026-08-19",
  "ece_date_min": "2026-07-20",
  "ece_rows": 150,
  "ece_stations": [
    "ECE_BBG_Lost_Meadow",
    "ECE_BBG_Main_St",
    "ECE_Renton_Garden_North",
    "ECE_Renton_Garden_Shed",
    "ECE_Renton_Home"
  ],
  "ece_target_used_for_fit": false,
  "train_rows": 9803,
  "trainval_rows": 14608,
  "val_rows": 4805,
  "wa_test_rows": 6620,
  "wa_train_stations": [
    "BeaverPass_WA_990",
    "CayusePass_WA",
    "Darrington",
    "Paradise_WA",
    "Quinault",
    "SourdoughGulch_WA_985",
    "Spokane"
  ]
}
REPORT_END::INPUT_AUDIT
REPORT_BEGIN::FEATURE_AUDIT
| component                    |   parent_count |   dropped_smap |   effective_count | effective_features                                                                                                                                                                                                                                                                                 

## Router audit

Regime shares are shown for WA trainval, the held-out WA temporal test, and the unseen ECE sensor set.

In [3]:
print("REPORT_BEGIN::ROUTER_AUDIT")
print(audit.to_markdown(index=False, floatfmt=".4f"))
print("REPORT_END::ROUTER_AUDIT")

REPORT_BEGIN::ROUTER_AUDIT
| model_id                         | router       | dataset     |   router_seed |     n |   regime_0_share |   regime_1_share |   router_feature_count | router_features                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           |
|:---------------------------------|:---

## Seven-model metrics

RMSE is the primary error metric; Pearson and first-difference Pearson correlation assess whether predictions follow the target level and temporal trend.

In [4]:
pooled = summary[summary["scope"] == "__pooled__"].copy()
columns = ["model_id", "dataset", "window", "n_seeds", "rmse_mean", "rmse_std", "mae_mean", "bias_mean", "ubrmse_mean", "r2_mean", "pearson_mean", "diff_pearson_mean"]
print("REPORT_BEGIN::METRICS")
print(pooled[columns].sort_values(["dataset", "window", "rmse_mean"]).to_markdown(index=False, floatfmt=".6f"))
print("REPORT_END::METRICS")

REPORT_BEGIN::METRICS
| model_id                         | dataset     | window               |   n_seeds |   rmse_mean |   rmse_std |   mae_mean |   bias_mean |   ubrmse_mean |   r2_mean |   pearson_mean |   diff_pearson_mean |
|:---------------------------------|:------------|:---------------------|----------:|------------:|-----------:|-----------:|------------:|--------------:|----------:|---------------:|--------------------:|
| Trained_Gating_k2_no_smap        | ece_spatial | spatial_ece_v3_full  |         1 |    0.047194 |   0.000000 |   0.034377 |   -0.001075 |      0.047181 | -0.006159 |      -0.056421 |           -0.124484 |
| Seasonal_Binary_k2_no_smap       | ece_spatial | spatial_ece_v3_full  |         1 |    0.078043 |   0.000000 |   0.070908 |    0.060941 |      0.048754 | -1.751515 |      -0.004370 |            0.077467 |
| Univariate_G_API_k2_no_smap      | ece_spatial | spatial_ece_v3_full  |         1 |    0.078843 |   0.000000 |   0.071636 |    0.061992 |      0.048

## Original-model comparison

The following table compares same-seed no-SMAP results with the original SMAP-trained reference runs; reference rows are never used for fitting.

In [5]:
comparison = pd.read_csv(EXP_DIR / "reference_comparison.csv", low_memory=False)
print("REPORT_BEGIN::REFERENCE_COMPARISON")
if comparison.empty:
    print("No reference rows available.")
else:
    columns = ["model_id", "seed", "dataset", "rmse_no_smap", "rmse_original", "rmse_delta_no_smap_minus_original", "pearson_no_smap", "pearson_original"]
    print(comparison[columns].to_markdown(index=False, floatfmt=".6f"))
print("REPORT_END::REFERENCE_COMPARISON")

REPORT_BEGIN::REFERENCE_COMPARISON
| model_id                         |   seed | dataset     |   rmse_no_smap |   rmse_original |   rmse_delta_no_smap_minus_original |   pearson_no_smap |   pearson_original |
|:---------------------------------|-------:|:------------|---------------:|----------------:|------------------------------------:|------------------:|-------------------:|
| Clustering_Backbone54_k2_no_smap |     42 | ece_spatial |       0.137922 |        0.057342 |                            0.080579 |          0.065928 |           0.109832 |
| Clustering_Backbone54_k2_no_smap |     42 | wa_temporal |       0.072340 |        0.043909 |                            0.028431 |          0.867413 |           0.904845 |
| Clustering_Dynamic_k2_no_smap    |     42 | ece_spatial |       0.082021 |        0.059859 |                            0.022162 |         -0.039857 |          -0.140396 |
| Clustering_Dynamic_k2_no_smap    |     42 | wa_temporal |       0.067758 |        0.047057 | 

## SMAP-invariance check

Seed-42 ECE rows are evaluated once with native SMAP values and once with all SMAP columns replaced by zero; a no-SMAP model must produce identical regimes and predictions.

In [6]:
invariance = pd.read_csv(EXP_DIR / "smap_invariance.csv", low_memory=False)
print("REPORT_BEGIN::SMAP_INVARIANCE")
print(invariance.to_markdown(index=False, floatfmt=".12f"))
print("REPORT_END::SMAP_INVARIANCE")

REPORT_BEGIN::SMAP_INVARIANCE
| model_id                         |   seed |   smap_columns_altered |   max_abs_prediction_difference |   changed_regime_labels |
|:---------------------------------|-------:|-----------------------:|--------------------------------:|------------------------:|
| Clustering_V0_Full_k2_no_smap    |     42 |                     85 |                  0.000000000000 |                       0 |
| Clustering_Backbone54_k2_no_smap |     42 |                     85 |                  0.000000000000 |                       0 |
| Trained_Gating_k2_no_smap        |     42 |                     85 |                  0.000000000000 |                       0 |
| Univariate_G_API_k2_no_smap      |     42 |                     85 |                  0.000000000000 |                       0 |
| Clustering_Dynamic_k2_no_smap    |     42 |                     85 |                  0.000000000000 |                       0 |
| Seasonal_Binary_k2_no_smap       |     42 |        

## Trend figures

The next cell generates two five-line figure suites per ECE station: architecture gates and the three alternative regime gates, each against observed target values.

In [7]:
figure_paths = runner.make_trend_figures(predictions, EXP_DIR / "figures")
print("REPORT_BEGIN::FIGURES")
for path in figure_paths:
    print(f"- {path.name}")
print("REPORT_END::FIGURES")

REPORT_BEGIN::FIGURES
- ece_ECE_BBG_Lost_Meadow_architecture_trend.png
- ece_ECE_BBG_Lost_Meadow_regime_trend.png
- ece_ECE_BBG_Main_St_architecture_trend.png
- ece_ECE_BBG_Main_St_regime_trend.png
- ece_ECE_Renton_Garden_North_architecture_trend.png
- ece_ECE_Renton_Garden_North_regime_trend.png
- ece_ECE_Renton_Garden_Shed_architecture_trend.png
- ece_ECE_Renton_Garden_Shed_regime_trend.png
- ece_ECE_Renton_Home_architecture_trend.png
- ece_ECE_Renton_Home_regime_trend.png
REPORT_END::FIGURES


## Completion

All tables above are sourced from executed cells, and the linked figures are generated during this notebook execution.

In [8]:
print("Notebook complete — all report sections above are generated from experiment artifacts.")

Notebook complete — all report sections above are generated from experiment artifacts.
